Notebook Header & Metadata

#  NLP Task 3 – Chatbot using Hugging Face Transformers

## Objective
To build a conversational chatbot using a pre-trained transformer model from Hugging Face that can generate human-like responses.

## Tools Used
- Python
- Hugging Face Transformers
- PyTorch
- Google Colab

**Install Required Libraries**

In [ ]:
# INSTALL DEPENDENCIES
# Run this only once per environment
!pip install transformers torch accelerate -q

print("✅ Dependencies installed successfully!")

✅ Dependencies installed successfully!


**Import Libraries**

In [ ]:
# IMPORT NECESSARY LIBRARIES
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import sys

print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ Transformers library ready")

✅ PyTorch version: 2.10.0+cpu
✅ Transformers library ready


**Load Pre-trained Model & Tokenizer **

In [ ]:
# MODEL LOADING - MANDATORY STEP
# Using DialoGPT-medium for balance of performance & memory usage
MODEL_NAME = "microsoft/DialoGPT-medium"  # Options: -small, -medium, -large

print(f"🔄 Loading model: {MODEL_NAME} ...")

# Load tokenizer and model from Hugging Face Hub
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# Set model to evaluation mode
model.eval()

print("✅ Model and tokenizer loaded successfully!")
print(f"✅ Model device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

🔄 Loading model: microsoft/DialoGPT-medium ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✅ Model and tokenizer loaded successfully!
✅ Model device: CPU


**Define Response Generation Function**

In [ ]:
# RESPONSE GENERATION FUNCTION
def generate_bot_response(user_input, chat_history_ids, step):
    """
    Generate chatbot response using DialoGPT model

    Args:
        user_input (str): User's message
        chat_history_ids: Previous conversation tensor
        step (int): Current conversation turn number

    Returns:
        tuple: (bot_response_str, updated_chat_history_ids)
    """
    # Encode user input + EOS token
    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors='pt'
    )

    # Concatenate with chat history (if not first turn)
    bot_input_ids = torch.cat(
        [chat_history_ids, new_input_ids],
        dim=-1
    ) if step > 0 else new_input_ids

    # Generate response with sampling for better diversity
    # Parameters tuned for conversational quality
    chat_history_ids = model.generate(
        bot_input_ids,
        max_length=1000,              # Max tokens in response
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,               # Enable sampling (not greedy)
        top_k=50,                     # Sample from top 50 tokens
        top_p=0.95,                   # Nucleus sampling threshold
        temperature=0.7,              # Control randomness (0.7 = balanced)
        num_return_sequences=1        # Single response
    )

    # Decode and extract only the NEW response (skip history)
    bot_response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    return bot_response, chat_history_ids

In [ ]:
generate_bot_response("Hello", None, 0)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


('Howdy! How are you?',
 tensor([[15496, 50256,  2437,  9892,  5145,  1374,   389,   345,  5633, 50256]]))

** Main Chat Loop Function**

In [ ]:
# MAIN CHATBOT FUNCTION - CONVERSATION FLOW
def run_chatbot():
    """
    Interactive console chatbot with exit condition
    """
    print("\n" + "="*60)
    print("🤖 WELCOME TO YOUR AI ASSISTANT")
    print("="*60)
    print("💬 Type your message and press Enter")
    print("🚪 Type 'exit' or 'quit' to end conversation")
    print("="*60 + "\n")

    # Initialize chat history
    chat_history_ids = None
    step = 0

    while True:
        # Get user input
        user_input = input("👤 You: ").strip()

        # Exit condition check
        if user_input.lower() in ['exit', 'quit', 'bye', 'goodbye']:
            print("\n🤖 Chatbot: Thank you for chatting! Have a great day! 👋")
            print("="*60 + "\n")
            break

        # Skip empty inputs
        if not user_input:
            continue

        # Generate and display bot response
        try:
            bot_response, chat_history_ids = generate_bot_response(
                user_input,
                chat_history_ids,
                step
            )
            print(f"🤖 Chatbot: {bot_response}\n")
            step += 1

        except Exception as e:
            print(f"❌ Error generating response: {e}")
            print("💡 Tip: Try rephrasing your question\n")

In [ ]:
run_chatbot()


🤖 WELCOME TO YOUR AI ASSISTANT
💬 Type your message and press Enter
🚪 Type 'exit' or 'quit' to end conversation

